In [1]:
# ============================================================
# CELDA 1 · Importar librerías y cargar el dataset
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')  # Oculta avisos que no son errores

# Cargar el dataset desde la carpeta data/raw/
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Configuración visual para las gráficas
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Primera vista del dataset
print("=== DIMENSIONES DEL DATASET ===")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

print("\n=== PRIMERAS 3 FILAS ===")
print(df.head(3).to_string())

print("\n=== TIPOS DE DATOS DE CADA COLUMNA ===")
print(df.dtypes)

=== DIMENSIONES DEL DATASET ===
Filas: 7043 | Columnas: 21

=== PRIMERAS 3 FILAS ===
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling     PaymentMethod  MonthlyCharges TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes  Electronic check           29.85        29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No      Mailed check           56.95       1889.5    No
2  3668-QPYBK    Male              0      No         No       2      

In [2]:
# ============================================================
# CELDA 2 · Estadísticas básicas y análisis de nulos
# ============================================================

print("=== ESTADÍSTICAS BÁSICAS (columnas numéricas) ===")
print(df.describe().to_string())

print("\n=== VALORES NULOS POR COLUMNA ===")
nulos = df.isnull().sum()
print(nulos[nulos >= 0])  # Muestra todas las columnas

print("\n=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (Churn) ===")
print(df['Churn'].value_counts())
print(f"\nPorcentaje de Churn:")
print(df['Churn'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

=== ESTADÍSTICAS BÁSICAS (columnas numéricas) ===
       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000

=== VALORES NULOS POR COLUMNA ===
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dty

In [3]:
# ============================================================
# CELDA 3 · Reporte automático con ydata-profiling
# ============================================================
import os
from ydata_profiling import ProfileReport

# Generar el reporte completo (tarda 1-2 minutos)
print("Generando reporte... esto puede tardar 1-2 minutos")

profile = ProfileReport(
    df,
    title="Telco Churn - EDA Report",
    explorative=True  # Análisis más profundo
)

# Guardar el reporte como HTML en la carpeta reports/
os.makedirs('../reports', exist_ok=True)
profile.to_file('../reports/eda_report.html')

print("✅ Reporte guardado en reports/eda_report.html")
print("Ábrelo en el navegador para verlo completo")

Generando reporte... esto puede tardar 1-2 minutos


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Reporte guardado en reports/eda_report.html
Ábrelo en el navegador para verlo completo


In [4]:
# ============================================================
# CELDA 4 · Visualizaciones clave para el portfolio
# ============================================================
import os
os.makedirs('../reports/figures', exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Telco Churn - Análisis Exploratorio', fontsize=16, fontweight='bold')

# --- GRÁFICA 1: Distribución del Churn (variable objetivo) ---
churn_counts = df['Churn'].value_counts()
axes[0,0].pie(
    churn_counts.values,
    labels=['No Churn', 'Churn'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90
)
axes[0,0].set_title('Distribución del Churn')

# --- GRÁFICA 2: Churn por tipo de contrato ---
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
contract_churn.columns = ['Contract', 'ChurnRate']
axes[0,1].bar(contract_churn['Contract'], contract_churn['ChurnRate'],
              color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0,1].set_title('Tasa de Churn por Tipo de Contrato')
axes[0,1].set_ylabel('% Churn')
axes[0,1].tick_params(axis='x', rotation=15)

# --- GRÁFICA 3: Distribución de la antigüedad (tenure) ---
axes[0,2].hist(
    [df[df['Churn']=='No']['tenure'], df[df['Churn']=='Yes']['tenure']],
    bins=30, label=['No Churn', 'Churn'],
    color=['#2ecc71', '#e74c3c'], alpha=0.7
)
axes[0,2].set_title('Antigüedad del Cliente (meses)')
axes[0,2].set_xlabel('Meses')
axes[0,2].legend()

# --- GRÁFICA 4: Cargo mensual vs Churn ---
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1,0],
           patch_artist=True)
axes[1,0].set_title('Cargo Mensual por Churn')
axes[1,0].set_xlabel('Churn')
axes[1,0].set_ylabel('USD/mes')
plt.sca(axes[1,0])
plt.title('Cargo Mensual por Churn')

# --- GRÁFICA 5: Churn por método de pago ---
payment_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
payment_churn.columns = ['PaymentMethod', 'ChurnRate']
payment_churn['PaymentMethod'] = payment_churn['PaymentMethod'].str.replace(
    ' (automatic)', '\n(auto)', regex=False
)
axes[1,1].barh(payment_churn['PaymentMethod'], payment_churn['ChurnRate'],
               color='#9b59b6')
axes[1,1].set_title('Tasa de Churn por Método de Pago')
axes[1,1].set_xlabel('% Churn')

# --- GRÁFICA 6: Correlación de variables numéricas ---
numeric_cols = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
# TotalCharges viene como texto en este dataset, hay que convertirla
numeric_cols['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'], errors='coerce'
)
numeric_cols['Churn_num'] = (df['Churn'] == 'Yes').astype(int)
corr_matrix = numeric_cols.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1,2], square=True)
axes[1,2].set_title('Correlación entre Variables Numéricas')

plt.tight_layout()
plt.savefig('../reports/figures/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada en reports/figures/eda_overview.png")

✅ Gráfica guardada en reports/figures/eda_overview.png


In [5]:
# ============================================================
# CELDA 5 · Valores no numéricos para TotalCharges
# ============================================================

print("\n--- TotalCharges (¿hay valores no numéricos?) ---")
print(f"Valores no numéricos en TotalCharges: "
      f"{pd.to_numeric(df['TotalCharges'], errors='coerce').isnull().sum()}")


--- TotalCharges (¿hay valores no numéricos?) ---
Valores no numéricos en TotalCharges: 11


In [6]:
# ============================================================
# CELDA 6 · Correlación entre Churn y las 11 variables
# (versión actualizada con InternetService)
# ============================================================

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle('Tasa de Churn para las variables: Partner, Dependents, MultipleLines, PaperlessBilling, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies e InternetService', 
             fontsize=15, fontweight='bold', y=1.01)

variables = [
    'Partner', 'Dependents', 'MultipleLines', 'PaperlessBilling',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'InternetService'
]

for idx, var in enumerate(variables):
    ax = axes[idx // 4][idx % 4]
    
    # Calcular tasa de churn por categoría
    churn_rate = df.groupby(var)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).reset_index()
    churn_rate.columns = [var, 'ChurnRate']
    churn_rate = churn_rate.sort_values('ChurnRate', ascending=True)
    
    # Color según nivel de churn
    bar_colors = []
    for rate in churn_rate['ChurnRate']:
        if rate < 20:
            bar_colors.append('#2ecc71')
        elif rate < 35:
            bar_colors.append('#f39c12')
        else:
            bar_colors.append('#e74c3c')
    
    bars = ax.barh(churn_rate[var], churn_rate['ChurnRate'], color=bar_colors)
    
    # Etiquetas de porcentaje
    for bar, rate in zip(bars, churn_rate['ChurnRate']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{rate:.1f}%', va='center', fontsize=9, fontweight='bold')
    
    ax.set_title(var, fontweight='bold', fontsize=11)
    ax.set_xlabel('% Churn', fontsize=9)
    ax.set_xlim(0, 60)
    
    # Línea de referencia = media global 26.5%
    ax.axvline(x=26.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(26.5, -0.6, 'media\n26.5%', fontsize=7, color='gray', ha='center')

# Ocultar el único subplot vacío (posición 3,4 → índice [2][3])
axes[2][3].set_visible(False)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Churn < 20% · bajo riesgo'),
    Patch(facecolor='#f39c12', label='Churn 20-35% · riesgo medio'),
    Patch(facecolor='#e74c3c', label='Churn > 35% · alto riesgo'),
]
fig.legend(handles=legend_elements, loc='lower right', 
           bbox_to_anchor=(0.95, 0.05), fontsize=10,
           frameon=True, fancybox=True)

plt.tight_layout()
plt.savefig('../reports/figures/churn_variables_adicionales.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica actualizada en reports/figures/churn_variables_adicionales.png")

✅ Gráfica actualizada en reports/figures/churn_variables_adicionales.png


### Observaciones sobre cada variable ###  

- **Churn** está desbalanceada como se puede apreciar en el gráfico de distribución del churn.
- **Contract:** la gráfica de la tasa de Churn por tipo de contrato refleja que los clientes con un contrato mes a mes abandonan al ~43% mientras que los de un año bajan al ~11% y los de dos años casi no se van. Parece que el tipo de contrato va a ser un predictor importante.
- **PhoneService** está muy desbalanceada (EDA Report).
- **customerID** tiene valores únicos, por lo que no es una variable predictiva (EDA Report). 
- **tenure:** como se aprecia en el histograma del EDA Report (lo corrobora la kurtosis negativa), presenta una distribución plana entre 0 y 72 meses. Tiene 11 valores a cero. En la gráfica de tipo de Churn por tipo de contrato se observa que los clientes que tienen menos antigüedad son los que más se van y si llevan más de 3 años prácticamente no se van. Tenure será un predictor importante.
- **MonthlyCharges:** también distribución plana. Sin valores a cero. En la gráfica de cargo mensual por Churn se ve que los clientes que se van tienen un valor de mediana de MonthlyCharges más alta que los que se quedan. Tiene sentido que los que tienen la cuota más alta tienen más probabilidad de encontrar alternativas más baratas.
- **TotalCharges:** se está leyendo como texto pero en realidad es un número. Además, presenta 11 valores no numéricos. En la gráfica de correlación se puede comprobar que TotalCharges correlaciona mucho con tenure.
- **gender** no parece que vaya a aportar demasiado al modelo, está muy equilibrado.
- **SeniorCitizen:** solo el 16,2% están categorizados como ciudados senior. En el dataset aparece como de tipo numérico, pero en realidad es categórica (0 no es mayor / 1 sí es mayor).
- **PaymentMethod:** en la gráfica de tasa de Churn según método de pago se puede ver que los que pagan con cheque electrónico abandonan casi al ~45%, casi el triple que los que pagan mediante métodos automáticos. El cheque electrónico supone realizar una acción manual, por lo que pueden ser clientes con más fricción o menos compromiso.
- **Partner:** tener pareja supone un ancla de retención como se puede observar en la gráfica.
- **Dependents:** un cliente con familia tiene más resistencia a cambiar su línea.
- **OnlineSecurity, TechSupport, OnlineBackup y DeviceProtection:** los clientes que no tienen ninguno de estos servicios son los que más cerca están de marcharse. Contratar alguno de estos servicios reduce a la mitad el riesgo de que se vaya un cliente. Los clientes que no tienen internet contratado son los más fieles, no tienen tan fácil la comparación ni la recepción de ofertas.
- **MultipleLines** presenta diferencias mínimas entre las tres opciones.
- **PaperlessBilling** nos confirma que los clientes más digitalizados se van el doble que los más analógicos.
- **StreamingTV y StreamingMovies:** tener o no tener estos servicios no presenta diferencias respecto al Churn.
- **InternetService:** los clientes con fibra óptica tienen más del doble de probabilidad de irse que los de DSL.

# CONCLUSIONES #  

- Utilizar un parámetro a la hora de preprocesar que indique al modelo que los casos de Yes para Churn pesen más que los de No. Esto tratará de solventar el problema de desbalanceo de Churn.
- Tratar como categórica la variable SeniorCitizen.
- Se podría convertir TotalCharges a numérico y solventar el problema de los valores no numéricos. Su valor tendría que ser más o menos tenure*MonthlyCharges (el total facturado al cliente). Pero según el heatmap, presenta una correlación alta con tenure, por lo que la eliminaremos.
- MonthlyCharges necesita escalado porque el rango es grande (min=18.25, media=64.76, max=118.75).
- customerID se eliminará.
- Habría que estudiar si es más rentable ofrecer gratis por tiempo los servicios de OnlineSecurity/TechSupport/OnlineBackup/DeviceProtection (a los clientes que no los tengan) que el coste de perder a los clientes.
- MultipleLines no ayuda a discriminar el Churn.
- Todas las columnas muestran 0 nulos reales, lo que es inusual.
- Con lo que se ha visto podríamos concluir que Contract, InternetService, tenure, OnlineSecurity, TechSupport, PaymentMethod, MonthlyCharges, OnlineBackup y DeviceProtection son las variables que más van a servir para hacer la predicción. Por contra, gender, PhoneService, customerID y TotalCharges no aportan poder predictivo.